# Notifications Persuade the Already Persuaded — reproducibility notebook (Drink Less MRT)

Secondary analysis of the open data of the **Drink Less micro-randomized trial** (Bell et al., 2023, *JMIR mHealth uHealth* 11:e38342, https://doi.org/10.2196/38342; data: Garnett et al., OSF https://osf.io/w3szp).
349 new users of an alcohol-reduction app were randomized **every day at 8 pm** (days 1–30) to receive a notification (p = 0.6: standard 0.3, new message 0.3) or no notification (0.4). Outcome: **opening the app between 8 and 9 pm**.

**How to use (Google Colab):** `Runtime → Run all` (≈ 5–8 min). The notebook downloads the OSF data automatically; if that fails, it asks you to upload `w3szp-osfstorage-archive.zip` (the *Download As Zip* file from OSF). All figures are displayed inline, and at the end **`drinkless_results.zip`** is downloaded with figures, tables and `RESULTS_SUMMARY.md`.

| Section | Content |
|---|---|
| 0 | Setup and data download |
| 1 | Data preparation (weekday reconstructed from Dataset B) |
| 2 | Descriptives |
| 3 | Causal excursion effects (WCLS, absolute scale) and moderators |
| 4 | Effect by week: absolute (risk difference) vs relative (risk ratio) |
| 5 | Engagement-state moderation |
| 6 | CATE models (M1–M5) with leave-users-out cross-fitting, budget-matched evaluation, calibration |
| 6b | Paired bootstrap comparison of models (difference in area under G(b)) |
| 7 | Robustness across 4 random user partitions |
| 8 | Explainable rules (linear coefficients, 2-variable rule) and SHAP |
| 9 | Equity check: who receives the notifications under tailoring |
| 10 | Export results |

## 0. Setup and data

In [ ]:
import sys, os, subprocess, shutil, zipfile, urllib.request, glob, json, time, warnings, platform
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "shap", "statsmodels"], check=True)
T0 = time.time()
OUT = "drinkless_results"; os.makedirs(OUT, exist_ok=True)
ZIP = "w3szp-osfstorage-archive.zip"
if not os.path.exists(ZIP):
    try:
        print("Downloading data from OSF ...")
        urllib.request.urlretrieve("https://files.osf.io/v1/resources/w3szp/providers/osfstorage/?zip=", ZIP)
        zipfile.ZipFile(ZIP).testzip()
    except Exception as e:
        print("Automatic download failed:", e)
        if os.path.exists(ZIP): os.remove(ZIP)
        if IN_COLAB:
            from google.colab import files
            print("Please upload w3szp-osfstorage-archive.zip (OSF > Files > Download As Zip)")
            up = files.upload(); name = list(up.keys())[0]
            if name != ZIP: os.rename(name, ZIP)
        else:
            raise SystemExit("Place w3szp-osfstorage-archive.zip next to this notebook.")
shutil.rmtree("osf_data", ignore_errors=True)
zipfile.ZipFile(ZIP).extractall("osf_data")
rds = sorted(glob.glob("osf_data/**/*Dataset_*.rds", recursive=True)); print([os.path.basename(r) for r in rds])

In [ ]:
# The .rds files are converted to CSV with R (pre-installed in Colab; installed if missing).
if shutil.which("Rscript") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y r-base-core > /dev/null", shell=True, check=True)
dA = [r for r in rds if r.endswith("Dataset_A.rds")][0]; dB = [r for r in rds if r.endswith("Dataset_B.rds")][0]
rcode = f'''
a <- as.data.frame(readRDS("{dA}")); a$message <- NULL; write.csv(a, "ds_A.csv", row.names = FALSE)
b <- readRDS("{dB}"); b <- as.data.frame(b[, c("ID", "downloaded_atDAY", "localScheduledDay", "days_since_download", "subversion")])
write.csv(b, "ds_B.csv", row.names = FALSE)
'''
open("convert.R", "w").write(rcode)
r = subprocess.run(["Rscript", "convert.R"], capture_output=True, text=True); print(r.returncode, r.stderr[-300:] if r.returncode else "converted")

In [ ]:
import numpy as np, pandas as pd, statsmodels.api as sm, lightgbm as lgb, shap, sklearn
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
import matplotlib; import matplotlib.pyplot as plt
from IPython.display import display, Image as _Img, Markdown
warnings.filterwarnings("ignore")
P = 0.6
RES = {"env": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "lightgbm": lgb.__version__,
               "sklearn": sklearn.__version__, "shap": shap.__version__, "statsmodels": sm.__version__}}
INK2, MUTED, GRID = "#52514e", "#898781", "#e1e0d9"
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": MUTED, "axes.labelcolor": INK2, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
                     "grid.linewidth": 0.6, "axes.axisbelow": True, "legend.frameon": False})
def show(fig, name):
    fig.tight_layout(); fig.savefig(f"{OUT}/{name}.png", dpi=300); plt.show(); plt.close(fig)
print(RES["env"])

## 1. Data preparation
Unit: user-day (days 1–30). Calendar date = download date + days since download (Dataset B), used to define weekends.

In [ ]:
A = pd.read_csv("ds_A.csv"); B = pd.read_csv("ds_B.csv")
assert ((B.localScheduledDay - B.downloaded_atDAY) == B.days_since_download).all()
dl = B.groupby("ID").downloaded_atDAY.first()
d = A.rename(columns={"days_since_download": "day", "treatment": "A", "primary_outcome": "y"}).copy()
date = pd.to_datetime(d.ID.map(dl) + d.day, unit="D", origin="1970-01-01")
d["date"] = date
d["weekend"] = date.dt.dayofweek.isin([5, 6]).astype(int)
d["female"] = d.gender; d["manual"] = (d.employment_type == 1).astype(int)
d = d.reset_index(drop=True)
CTX = ["day", "weekend", "before_8pm", "after_9pm_day_before", "already_engaged", "habituation"]
TRAITS = ["age", "female", "manual", "AUDIT_score"]
ALL = CTX + TRAITS
users = d.ID.unique(); G = d.ID.values; iu = {u: np.where(G == u)[0] for u in users}
print(d.shape, "users:", len(users), "| P(notification) =", round(d.A.mean(), 3), "| dates:", date.min().date(), "to", date.max().date())
d[["ID", "day", "date", "weekend", "A", "subversion", "y"] + CTX[2:] + TRAITS].head()

## 2. Descriptives

In [ ]:
ud = d.groupby("ID").first()
desc = dict(n_user_days=len(d), n_users=int(len(users)), p_notification=round(d.A.mean(), 3), p_standard=round((d.subversion == "Y").mean(), 3),
            p_new=round((d.subversion == "X").mean(), 3), p_open_8_9pm=round(d.y.mean(), 3),
            p_open_if_notified=round(d.y[d.A == 1].mean(), 3), p_open_if_not=round(d.y[d.A == 0].mean(), 3),
            date_first=str(date.min().date()), date_last=str(date.max().date()), weekend_share=round(d.weekend.mean(), 3),
            age_mean=round(ud.age.mean(), 1), age_sd=round(ud.age.std(), 1), age_range=[int(ud.age.min()), int(ud.age.max())],
            female=round(ud.female.mean(), 3), manual=round(ud.manual.mean(), 3), audit_mean=round(ud.AUDIT_score.mean(), 1),
            audit_sd=round(ud.AUDIT_score.std(), 1), audit_cat=ud.AUDIT_score_cat.value_counts().to_dict(),
            p_used_before_8pm=round(d.before_8pm.mean(), 3), p_already_engaged=round(d.already_engaged.mean(), 3))
RES["descriptives"] = desc
d[ALL + ["y"]].describe().T.round(3).to_csv(f"{OUT}/table_descriptives.csv")
print(json.dumps(desc, indent=1, default=str))

# Fig 1: daily probability of opening the app, with vs without notification
g = d.groupby(["day", "A"]).y.mean().unstack()
fig, ax = plt.subplots(figsize=(5.6, 3.0))
ax.plot(g.index, g[1], "-o", ms=3, color=C[1], lw=1.6, label="Notification sent")
ax.plot(g.index, g[0], "-o", ms=3, color=C[0], lw=1.6, label="No notification")
ax.set_xlabel("Days since download"); ax.set_ylabel("P(open app 8–9 pm)"); ax.set_ylim(0, None); ax.legend(fontsize=8, labelcolor=INK2)
show(fig, "fig1_daily_open_rates")

## 3. Causal excursion effects (WCLS, absolute scale)
Linear model of Y on controls and (A − 0.6)·f(X); the centred treatment makes the coefficients causal (risk differences) even if the control model is misspecified. Cluster-robust SEs by user.

In [ ]:
def wcls(mods, controls=("day", "before_8pm", "already_engaged")):
    X = pd.DataFrame({"const": 1.0}, index=d.index)
    for c in controls: X[c] = d[c]
    c_ = d.A - P; X["A_c"] = c_
    for m in mods: X[f"A_c:{m}"] = c_ * d[m]
    f = sm.OLS(d.y, X).fit(cov_type="cluster", cov_kwds={"groups": d.ID}); ci = f.conf_int()
    return {k: dict(est=round(f.params[k], 4), se=round(f.bse[k], 4), lo=round(ci.loc[k, 0], 4), hi=round(ci.loc[k, 1], 4), p=round(f.pvalues[k], 4))
            for k in X.columns if k.startswith("A_c")}
base = ("day", "weekend", "before_8pm", "already_engaged")
W = {"marginal": wcls([]), "day": wcls(["day"]), "day_weekend": wcls(["day", "weekend"], base), "weekend": wcls(["weekend"], base),
     "before_8pm": wcls(["before_8pm"]), "already_engaged": wcls(["already_engaged"]),
     "after_9pm_day_before": wcls(["after_9pm_day_before"], ("day", "before_8pm", "already_engaged", "after_9pm_day_before")),
     "habituation": wcls(["habituation"], ("day", "before_8pm", "already_engaged", "habituation")),
     "age": wcls(["age"]), "female": wcls(["female"]), "AUDIT": wcls(["AUDIT_score"])}
X_ = pd.DataFrame({"const": 1.0, "day": d.day, "before_8pm": d.before_8pm, "already_engaged": d.already_engaged})
X_["new_c"] = (d.subversion == "X") - 0.3; X_["std_c"] = (d.subversion == "Y") - 0.3
ft = sm.OLS(d.y, X_).fit(cov_type="cluster", cov_kwds={"groups": d.ID}); ci = ft.conf_int()
W["type"] = {k: dict(est=round(ft.params[k], 4), se=round(ft.bse[k], 4), lo=round(ci.loc[k, 0], 4), hi=round(ci.loc[k, 1], 4), p=round(ft.pvalues[k], 4)) for k in ["new_c", "std_c"]}
RES["wcls"] = W
rows = [dict(model=m, term=k, **v) for m, dd in W.items() for k, v in dd.items()]
pd.DataFrame(rows).to_csv(f"{OUT}/table_wcls.csv", index=False); pd.DataFrame(rows)

## 4. Effect by week: absolute (risk difference) vs relative (risk ratio)

In [ ]:
rng = np.random.default_rng(3)
wk = np.minimum((d.day.values - 1) // 7, 3)
def by_week(ix):
    yy, aa, ww = d.y.values[ix], d.A.values[ix], wk[ix]; rd, rr, p1, p0 = [], [], [], []
    for w in range(4):
        m = ww == w; a1 = yy[m & (aa == 1)].mean(); a0 = yy[m & (aa == 0)].mean()
        rd.append(a1 - a0); rr.append(a1 / a0 if a0 > 0 else np.nan); p1.append(a1); p0.append(a0)
    return np.array(rd), np.array(rr), np.array(p1), np.array(p0)
rd, rr, p1, p0 = by_week(np.arange(len(d)))
Bs = [by_week(np.concatenate([iu[u] for u in rng.choice(users, len(users))])) for _ in range(2000)]
Brd = np.array([b[0] for b in Bs]); Brr = np.array([b[1] for b in Bs])
slope = lambda v: np.polyfit(np.arange(4), v, 1)[0]
s_rd = np.array([slope(b) for b in Brd]); s_lrr = np.array([slope(np.log(b)) for b in Brr if np.all(np.isfinite(np.log(b)))])
RES["by_week"] = [dict(week=w + 1, days=["1-7", "8-14", "15-21", "22-30"][w], p_notified=round(p1[w], 4), p_not=round(p0[w], 4),
                       rd=round(rd[w], 4), rd_lo=round(np.nanquantile(Brd[:, w], .025), 4), rd_hi=round(np.nanquantile(Brd[:, w], .975), 4),
                       rr=round(rr[w], 2), rr_lo=round(np.nanquantile(Brr[:, w], .025), 2), rr_hi=round(np.nanquantile(Brr[:, w], .975), 2)) for w in range(4)]
RES["trend"] = dict(rd_slope_per_week=round(slope(rd), 4), rd_lo=round(np.quantile(s_rd, .025), 4), rd_hi=round(np.quantile(s_rd, .975), 4),
                    rd_p_ge0=round((s_rd >= 0).mean(), 4), logrr_slope_per_week=round(slope(np.log(rr)), 4),
                    logrr_lo=round(np.quantile(s_lrr, .025), 4), logrr_hi=round(np.quantile(s_lrr, .975), 4))
pd.DataFrame(RES["by_week"]).to_csv(f"{OUT}/table_by_week.csv", index=False)
x = np.arange(1, 5); lab = ["Days 1–7", "8–14", "15–21", "22–30"]; Wk = RES["by_week"]
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.9))
ax[0].errorbar(x, rd, yerr=[[w["rd"] - w["rd_lo"] for w in Wk], [w["rd_hi"] - w["rd"] for w in Wk]], fmt="o", ms=6, color=C[0], elinewidth=1.4, mec="white", mew=1.5)
ax[0].set_ylim(0, max(w["rd_hi"] for w in Wk) * 1.15); ax[0].set_xticks(x); ax[0].set_xticklabels(lab)
ax[0].set_ylabel("Absolute effect\n(risk difference)"); ax[0].set_title("A. Absolute effect", fontsize=9, color=INK2, loc="left")
ax[1].errorbar(x, rr, yerr=[[w["rr"] - w["rr_lo"] for w in Wk], [w["rr_hi"] - w["rr"] for w in Wk]], fmt="o", ms=6, color=C[1], elinewidth=1.4, mec="white", mew=1.5)
ax[1].axhline(1, color=MUTED, lw=0.8); ax[1].set_ylim(0, max(w["rr_hi"] for w in Wk) * 1.1); ax[1].set_xticks(x); ax[1].set_xticklabels(lab)
ax[1].set_ylabel("Relative effect\n(risk ratio)"); ax[1].set_title("B. Relative effect", fontsize=9, color=INK2, loc="left")
show(fig, "fig2_effect_by_week")
print(RES["trend"])

## 5. Engagement-state moderation

In [ ]:
labels = {"before_8pm": "Used app earlier today", "already_engaged": "Opened app 8–9 pm yesterday",
          "after_9pm_day_before": "Used app after 9 pm yesterday", "habituation": "Notified yesterday", "weekend": "Weekend"}
rows = []
for v, lab_ in labels.items():
    for k in [0, 1]:
        m = d[v].values == k
        est = d.y[m & (d.A == 1)].mean() - d.y[m & (d.A == 0)].mean()
        bb = []
        for _ in range(1000):
            ix = np.concatenate([iu[u] for u in rng.choice(users, len(users))]); mm = d[v].values[ix] == k; yy = d.y.values[ix]; aa = d.A.values[ix]
            bb.append(yy[mm & (aa == 1)].mean() - yy[mm & (aa == 0)].mean())
        rows.append(dict(variable=v, label=lab_, value=k, n=int(m.sum()), rd=round(est, 4), lo=round(np.quantile(bb, .025), 4), hi=round(np.quantile(bb, .975), 4)))
mod = pd.DataFrame(rows); mod.to_csv(f"{OUT}/table_moderation_states.csv", index=False); RES["moderation_states"] = rows
fig, ax = plt.subplots(figsize=(5.6, 3.2)); yp = np.arange(len(labels))
for k, col, nm in [(0, C[0], "No"), (1, C[1], "Yes")]:
    s = mod[mod.value == k]
    ax.errorbar(s.rd, yp + (0.12 if k else -0.12), xerr=[s.rd - s.lo, s.hi - s.rd], fmt="o", ms=5, color=col, elinewidth=1.3, mec="white", mew=1.2, label=nm)
ax.set_yticks(yp); ax.set_yticklabels(list(labels.values())); ax.axvline(0, color=MUTED, lw=0.8); ax.grid(axis="y", visible=False)
ax.set_xlabel("Effect of notification (risk difference, 95% CI)"); ax.legend(title="State", fontsize=8, title_fontsize=8, labelcolor=INK2, loc="upper right")
show(fig, "fig3_engagement_moderation"); mod

## 6. CATE models with leave-users-out cross-fitting and budget-matched evaluation
M1 GBM (context + traits) · M2 GBM (context) · M3 Ridge (context, 6 variables) · M4 Ridge (day + weekend) · M5 Random forest (context). DR-learner with known p = 0.6; users split into 5 outer × 4 inner folds; 5 repetitions.

In [ ]:
Aarr, Y = d.A.values, d.y.values.astype(float)
prm_mu = dict(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=40, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1, n_jobs=2)
def nuis(tr, te):
    X = d[ALL]; m = lgb.LGBMRegressor(**prm_mu, random_state=1).fit(X.iloc[tr].assign(A=Aarr[tr]), Y[tr])
    return m.predict(X.iloc[te].assign(A=0)), m.predict(X.iloc[te].assign(A=1))
def pseudo(m0, m1, a, y): return m1 - m0 + (a - P) / (P * (1 - P)) * (y - np.where(a == 1, m1, m0))
MODELS = {"M1 GBM (context + traits)": ("gbm", ALL), "M2 GBM (context)": ("gbm", CTX), "M3 Ridge (context)": ("lin", CTX),
          "M4 Ridge (day + weekend)": ("lin", ["day", "weekend"]), "M5 Random forest (context)": ("rf", CTX)}
def fit_tau(kind, Xtr, phi, seed):
    if kind == "gbm":
        return lgb.LGBMRegressor(n_estimators=200, learning_rate=0.03, num_leaves=7, min_child_samples=150, subsample=0.8, subsample_freq=1,
                                 colsample_bytree=0.8, reg_lambda=5.0, verbose=-1, random_state=seed, n_jobs=2).fit(Xtr, phi)
    if kind == "rf":
        return make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=300, min_samples_leaf=100, max_features=0.5, random_state=seed, n_jobs=2)).fit(Xtr, phi)
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), RidgeCV(alphas=np.logspace(-1, 4, 30))).fit(Xtr, phi)
BUD = np.round(np.arange(0, 1.0001, 0.05), 2)
def gain_curve(psi, c, idx):
    s_, c_ = psi[idx], c[idx]; o = np.argsort(-c_); n = len(idx)
    cum = np.concatenate([[0], np.cumsum(s_[o])]) / n; ate = s_.mean()
    return np.array([cum[int(round(b * n))] - b * ate for b in BUD])
def run(seed=7, n_rep=5, B=1000):
    rng_ = np.random.default_rng(seed)
    cate = {k: np.zeros((n_rep, len(d))) for k in MODELS}; mu0 = np.zeros((n_rep, len(d))); mu1 = mu0.copy()
    for r in range(n_rep):
        perm = rng_.permutation(users); gm = {g: i for i, g in enumerate(perm)}; Gr = np.array([gm[g] for g in G])
        for tr, te in GroupKFold(5).split(d, groups=Gr):
            phi = np.zeros(len(tr))
            for itr, ite in GroupKFold(4).split(tr, groups=Gr[tr]):
                a_, b_ = nuis(tr[itr], tr[ite]); phi[ite] = pseudo(a_, b_, Aarr[tr[ite]], Y[tr[ite]])
            for k, (kind, F) in MODELS.items(): cate[k][r, te] = fit_tau(kind, d[F].iloc[tr], phi, r).predict(d[F].iloc[te])
            mu0[r, te], mu1[r, te] = nuis(tr, te)
    cate = {k: v.mean(0) for k, v in cate.items()}; psi = pseudo(mu0.mean(0), mu1.mean(0), Aarr, Y)
    idx = np.arange(len(d)); out, curves = {}, {}; i40, i60 = list(BUD).index(0.4), list(BUD).index(0.6)
    for k, c in cate.items():
        full = gain_curve(psi, c, idx)
        Bc = np.array([gain_curve(psi, c, np.concatenate([iu[u] for u in rng_.choice(users, len(users))])) for _ in range(B)]); ab = Bc.mean(1)
        q = pd.qcut(c, 4, labels=False)
        out[k] = dict(area=round(full.mean(), 4), area_lo=round(np.quantile(ab, .025), 4), area_hi=round(np.quantile(ab, .975), 4), p_area_le0=round((ab <= 0).mean(), 4),
                      g40=round(full[i40], 4), g40_lo=round(np.quantile(Bc[:, i40], .025), 4), g40_hi=round(np.quantile(Bc[:, i40], .975), 4),
                      g60=round(full[i60], 4), g60_lo=round(np.quantile(Bc[:, i60], .025), 4), g60_hi=round(np.quantile(Bc[:, i60], .975), 4),
                      cate_sd=round(c.std(), 4), share_pred_negative=round((c < 0).mean(), 3), quartile_effects=[float(round(psi[q == i].mean(), 4)) for i in range(4)])
        curves[k] = dict(est=full, lo=np.quantile(Bc, .025, 0), hi=np.quantile(Bc, .975, 0))
    return dict(summary=out, curves=curves, cate=cate, psi=psi, ate=round(psi.mean(), 4))
t1 = time.time(); MAIN = run(7); print("main run %.0fs" % (time.time() - t1))
ate = MAIN["ate"]; psi = MAIN["psi"]
tab = pd.DataFrame(MAIN["summary"]).T; tab.to_csv(f"{OUT}/table_models_main.csv"); tab

In [ ]:
# Fig 4: budget-matched gain curves
bud = BUD * 100; fig, ax = plt.subplots(figsize=(5.6, 3.9)); ax.axhline(0, color=MUTED, lw=0.8)
for i, (k, cv) in enumerate(MAIN["curves"].items()):
    if k.startswith(("M1", "M3")): ax.fill_between(bud, cv["lo"], cv["hi"], color=C[i], alpha=0.13, lw=0)
    ax.plot(bud, cv["est"], color=C[i], lw=2.3 if k.startswith("M3") else 1.4, label=k + (" ±95% CI" if k.startswith(("M1", "M3")) else ""))
ax.text(99, 0.0005, "Random targeting = 0", color=MUTED, fontsize=7, va="bottom", ha="right")
ax.set_xlabel("Notification budget (% of user-days notified)"); ax.set_ylabel("Gain over random targeting\n(risk difference per user-day)")
ax.set_xlim(0, 100); ax.legend(fontsize=7, loc="lower left", labelcolor=INK2, bbox_to_anchor=(0, 1.0))
show(fig, "fig4_budget_curves")

# Benefit retained at fixed budgets
share = {k: {f"{int(b*100)}%": round((b * ate + MAIN["curves"][k]["est"][i]) / ate, 3) for i, b in enumerate(BUD) if b in (0.2, 0.3, 0.4, 0.5, 0.6, 0.8)} for k in MODELS}
RES["budget_share"] = share; pd.DataFrame(share).T.to_csv(f"{OUT}/table_budget_share.csv"); display(pd.DataFrame(share).T)

# Fig 5: calibration (observed DR effect by quartile of predicted benefit)
fig, ax = plt.subplots(figsize=(5.0, 3.0)); qx = np.arange(1, 5)
for j, (k, off, col) in enumerate([("M1 GBM (context + traits)", -0.15, C[0]), ("M3 Ridge (context)", 0.0, C[2]), ("M4 Ridge (day + weekend)", 0.15, C[3])]):
    c = MAIN["cate"][k]; q = np.asarray(pd.qcut(c, 4, labels=False)); e = [psi[q == i].mean() for i in range(4)]
    Bq = np.array([[psi[ix][q[ix] == i].mean() for i in range(4)] for ix in (np.concatenate([iu[u] for u in rng.choice(users, len(users))]) for _ in range(1000))])
    ax.errorbar(qx + off, e, yerr=[np.array(e) - np.quantile(Bq, .025, 0), np.quantile(Bq, .975, 0) - np.array(e)], fmt="o", ms=5, color=col, elinewidth=1.3, mec="white", mew=1.2, label=k)
ax.axhline(ate, color=MUTED, lw=0.8, ls="--"); ax.set_xticks(qx); ax.set_xticklabels(["Q1\n(lowest)", "Q2", "Q3", "Q4\n(highest)"])
ax.set_xlabel("Quartile of predicted benefit (held-out users)"); ax.set_ylabel("Observed effect (DR)"); ax.legend(fontsize=7, labelcolor=INK2, loc="upper left")
show(fig, "fig5_calibration")

## 6b. Paired comparison of tailoring models
The per-model CIs in Section 6 use independent bootstrap draws. Here every model is evaluated on the **same** user-level bootstrap resamples, so the CI of the difference in area under G(b) (M3 minus each other model) accounts for the correlation between models. A CI that contains 0 means the data do not distinguish the two models; it does not by itself establish equivalence.

In [ ]:
rng_p = np.random.default_rng(2027); REF = "M3 Ridge (context)"; idx_all = np.arange(len(d))
boot_ix = [np.concatenate([iu[u] for u in rng_p.choice(users, len(users))]) for _ in range(1000)]
area_b = {k: np.array([gain_curve(psi, MAIN["cate"][k], ix).mean() for ix in boot_ix]) for k in MODELS}
area_0 = {k: gain_curve(psi, MAIN["cate"][k], idx_all).mean() for k in MODELS}
pair = []
for k in MODELS:
    if k == REF: continue
    dif = area_b[REF] - area_b[k]
    pair.append(dict(comparison=f"M3 - {k.split()[0]}", diff=round(area_0[REF] - area_0[k], 4), lo=round(np.quantile(dif, .025), 4),
                     hi=round(np.quantile(dif, .975), 4), p_two_sided=round(min(1.0, 2 * min((dif <= 0).mean(), (dif >= 0).mean())), 4)))
RES["paired_area_diff"] = pair; pd.DataFrame(pair).to_csv(f"{OUT}/table_paired_model_diff.csv", index=False); pd.DataFrame(pair)


## 7. Robustness: 3 additional random user partitions

In [ ]:
ROB = {7: MAIN["summary"]}
for sd in [11, 23, 101]: ROB[sd] = run(sd, B=500)["summary"]
rob = pd.DataFrame([dict(seed=s, model=k, **{x: v[x] for x in ["area", "area_lo", "area_hi", "p_area_le0", "g40", "g60"]}) for s, sm_ in ROB.items() for k, v in sm_.items()])
rob.to_csv(f"{OUT}/table_robustness.csv", index=False); RES["robustness"] = rob.to_dict("records")
fig, ax = plt.subplots(figsize=(5.6, 3.4)); ypos = 0; labs = []
for s in ROB:
    for i, k in enumerate(MODELS):
        v = ROB[s][k]; ax.errorbar(v["area"], -ypos, xerr=[[v["area"] - v["area_lo"]], [v["area_hi"] - v["area"]]], fmt="o", ms=4, color=C[i], elinewidth=1.1, label=k if s == 7 else None); ypos += 1
    labs.append((ypos - len(MODELS) / 2 - 0.5, f"seed {s}")); ypos += 1
ax.axvline(0, color=MUTED, lw=0.8); ax.set_yticks([-p for p, _ in labs]); ax.set_yticklabels([l for _, l in labs], fontsize=8); ax.grid(axis="y", visible=False)
ax.set_xlabel("Area under gain-over-random curve (95% CI)"); ax.legend(fontsize=6.5, labelcolor=INK2, loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2)
show(fig, "fig6_robustness"); rob.pivot_table(index="model", columns="seed", values="area")

## 8. Explainable rules and SHAP

In [ ]:
phi_all = np.zeros(len(d))
for tr, te in GroupKFold(5).split(d, groups=G):
    a_, b_ = nuis(tr, te); phi_all[te] = pseudo(a_, b_, Aarr[te], Y[te])
def raw_coefs(F):
    m = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 4, 30))).fit(d[F], phi_all)
    sc, rg = m.named_steps["standardscaler"], m.named_steps["ridgecv"]; coef = rg.coef_ / sc.scale_
    return m, float(rg.intercept_ - (rg.coef_ * sc.mean_ / sc.scale_).sum()), dict(zip(F, [float(c) for c in coef]))
m3, i3, c3 = raw_coefs(CTX); m4, i4, c4 = raw_coefs(["day", "weekend"])
thr3 = np.quantile(m3.predict(d[CTX]), 0.6); thr4 = np.quantile(m4.predict(d[["day", "weekend"]]), 0.6)
grid = pd.DataFrame({"day": np.tile(np.arange(1, 31), 2), "weekend": np.repeat([0, 1], 30)}); grid["pred"] = m4.predict(grid[["day", "weekend"]])
grid["send40"] = (grid.pred >= thr4).astype(int); grid.to_csv(f"{OUT}/table_rule_M4_grid.csv", index=False)
RES["rule_M3"] = dict(intercept=round(i3, 4), coefs={k: round(v, 4) for k, v in c3.items()}, threshold_40pct=round(float(thr3), 4))
RES["rule_M4"] = dict(intercept=round(i4, 4), coefs={k: round(v, 5) for k, v in c4.items()}, threshold_40pct=round(float(thr4), 4),
                      last_weekday=int(grid[(grid.weekend == 0) & (grid.send40 == 1)].day.max()) if grid[(grid.weekend == 0)].send40.any() else None,
                      last_weekend=int(grid[(grid.weekend == 1) & (grid.send40 == 1)].day.max()) if grid[(grid.weekend == 1)].send40.any() else None)
# share of user-days notified by M3 at 40% budget, by state
sel = m3.predict(d[CTX]) >= thr3
RES["rule_M3"]["notified_share_by_state"] = {v: {str(k): round(float(sel[d[v].values == k].mean()), 3) for k in [0, 1]} for v in ["before_8pm", "already_engaged", "weekend"]}
print(json.dumps({"M3": RES["rule_M3"], "M4": RES["rule_M4"]}, indent=1))
# Fig 7: M3 coefficients (effect of each variable on the expected benefit of a notification)
nice = {"day": "Day since download (per day)", "weekend": "Weekend", "before_8pm": "Used app earlier today", "after_9pm_day_before": "Used app after 9 pm yesterday",
        "already_engaged": "Opened app 8–9 pm yesterday", "habituation": "Notified yesterday"}
cs = pd.Series(c3).rename(nice).sort_values()
fig, ax = plt.subplots(figsize=(5.6, 2.9)); ax.barh(cs.index, cs.values, color=[C[1] if v < 0 else C[0] for v in cs.values], height=0.55)
ax.axvline(0, color=MUTED, lw=0.8); ax.grid(axis="y", visible=False); ax.set_xlabel("Change in predicted notification effect (risk difference)")
ax.set_xlim(min(cs.min(), 0) - 0.025, max(cs.max(), 0) + 0.025); ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(5))
for yv, v in enumerate(cs.values): ax.text(v + (0.002 if v >= 0 else -0.002), yv, f"{v:+.3f}", va="center", ha="left" if v >= 0 else "right", fontsize=7, color=INK2)
show(fig, "fig7_rule_M3_coefficients")
# Fig 8: SHAP of the flexible model M1
m1 = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.03, num_leaves=7, min_child_samples=150, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                       reg_lambda=5.0, verbose=-1, random_state=0, n_jobs=2).fit(d[ALL], phi_all)
SV = shap.TreeExplainer(m1).shap_values(d[ALL]); imp = pd.Series(np.abs(SV).mean(0), index=ALL).sort_values(ascending=False)
RES["shap_M1"] = imp.round(5).to_dict(); RES["shap_M1_traits_share"] = round(float(imp[TRAITS].sum() / imp.sum()), 3)
plt.figure(); shap.summary_plot(SV, d[ALL], show=False, max_display=10, plot_size=(5.6, 3.6))
plt.xlabel("SHAP value (impact on predicted notification effect)", fontsize=8); plt.tight_layout()
plt.savefig(f"{OUT}/fig8_shap_M1.png", dpi=300); plt.show(); plt.close("all")
print("Share of M1 attribution from user traits:", RES["shap_M1_traits_share"])

## 9. Equity check: who receives notifications under tailoring?
At a 40% budget, which users and states would the tailored rule (M3) notify, and how many users would receive no notification in the last two weeks?

In [ ]:
d["sel_M3"] = sel.astype(int)
eng = d.groupby("ID").apply(lambda g: g.before_8pm.mean() + g.already_engaged.mean())
tert = pd.qcut(eng.rank(method="first"), 3, labels=["Low", "Middle", "High"]); d["engagement_tertile"] = d.ID.map(tert)
eq = d.groupby("engagement_tertile").agg(share_notified_M3=("sel_M3", "mean"), share_notified_random=("A", "mean")).round(3)
late = d[d.day >= 17].groupby("ID").sel_M3.sum(); eq_zero = float((late == 0).mean())
RES["equity"] = dict(by_engagement_tertile=eq.reset_index().to_dict("records"), users_zero_notifications_days17_30=round(eq_zero, 3))
fig, ax = plt.subplots(figsize=(4.8, 2.8)); xx = np.arange(3)
ax.bar(xx - 0.18, eq.share_notified_random, width=0.36, color=C[0], label="Trial (random, 60%)")
ax.bar(xx + 0.18, eq.share_notified_M3, width=0.36, color=C[1], label="Tailored rule M3 (40% budget)")
ax.set_xticks(xx); ax.set_xticklabels([f"{t} engagement" for t in eq.index]); ax.set_ylabel("Share of user-days notified"); ax.grid(axis="x", visible=False)
ax.legend(fontsize=7.5, labelcolor=INK2)
show(fig, "fig9_equity"); print(RES["equity"])

## 10. Export results
Creates and downloads `drinkless_results.zip`.

In [ ]:
def clean(o):
    if isinstance(o, dict): return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, np.ndarray): return clean(o.tolist())
    if isinstance(o, np.generic): return o.item()
    return o
RES["ate_dr"] = ate; RES["models_main"] = MAIN["summary"]; RES["budgets"] = BUD.tolist()
RES["curves_main"] = {k: {kk: list(map(float, vv)) for kk, vv in v.items()} for k, v in MAIN["curves"].items()}
RES["runtime_sec"] = round(time.time() - T0, 1)
json.dump(clean(RES), open(f"{OUT}/results_all.json", "w"), indent=1)
f = lambda v: f"{v['est']:.4f} [{v['lo']:.4f}, {v['hi']:.4f}], p={v['p']:.4f}"
L = ["# Drink Less MRT — results summary", "", f"Runtime: {RES['runtime_sec']} s · Env: {RES['env']}", "",
     "## Descriptives", json.dumps(clean(desc)), "", "## WCLS (risk difference, open app 8–9 pm)",
     *[f"- {m}: " + "; ".join(f"{k}={f(v)}" for k, v in dd.items()) for m, dd in W.items()], "",
     f"## DR average effect: {ate}", "", "## By week", *[json.dumps(clean(r)) for r in RES["by_week"]], f"Trend: {clean(RES['trend'])}", "",
     "## Engagement-state moderation", *[json.dumps(clean(r)) for r in RES["moderation_states"]], "",
     "## Models (seed 7)", *[f"- {k}: {clean(v)}" for k, v in MAIN["summary"].items()], "",
     "## Paired model differences (area, M3 minus other)", *[json.dumps(clean(r)) for r in RES["paired_area_diff"]], "",
     "## Robustness (area by seed)", rob.pivot_table(index="model", columns="seed", values="area").to_string(), "",
     "## Benefit retained", pd.DataFrame(share).T.to_string(), "",
     f"## Rule M3: {clean(RES['rule_M3'])}", f"## Rule M4: {clean(RES['rule_M4'])}",
     f"## SHAP M1 (traits share {RES['shap_M1_traits_share']}): {clean(RES['shap_M1'])}", f"## Equity: {clean(RES['equity'])}"]
open(f"{OUT}/RESULTS_SUMMARY.md", "w").write("\n".join(L))
shutil.make_archive("drinkless_results", "zip", OUT)
print("\n".join(L)); print("\nFiles:", sorted(os.listdir(OUT)))
if IN_COLAB:
    from google.colab import files; files.download("drinkless_results.zip")